In [ ]:
import asyncio
import csv
import importlib
import inspect
import json
import time
from pathlib import Path

import loader as loader_pkg
import loader.crawl as crawl_pkg
import loader.crawl.detail_page as detail_page_module
import loader.crawl.performance_record as performance_record_module

importlib.reload(detail_page_module)
importlib.reload(performance_record_module)
importlib.reload(crawl_pkg)
loader_pkg = importlib.reload(loader_pkg)

crawl_detail_page = loader_pkg.crawl_detail_page
download_performance_record_pdf = loader_pkg.download_performance_record_pdf

print("crawl_detail_page 시그니처:", inspect.signature(crawl_detail_page))
print("download_performance_record_pdf 시그니처:", inspect.signature(download_performance_record_pdf))
assert "retry_count" in inspect.signature(download_performance_record_pdf).parameters


In [ ]:
SRC_DIR = Path.cwd().resolve()
PROJECT_DIR = SRC_DIR.parent
CSV_PATH = PROJECT_DIR / "raw" / "page_urls" / "search_scope.csv"
OUTPUT_DIR = SRC_DIR / "output" / "performance_record"
STATE_DIR = SRC_DIR / "output" / "_state"
CHECKPOINT_PATH = STATE_DIR / "crawl_checkpoint.json"

MAX_ITEMS = 10000
BATCH_SIZE = 2
DETAIL_CONCURRENCY = 2
PDF_CONCURRENCY = 1
LOG_EVERY = 10
PDF_RETRY_COUNT = 3
PDF_RETRY_BACKOFF_SECONDS = 2.0
BLOCKED_ERROR_THRESHOLD = 1

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
STATE_DIR.mkdir(parents=True, exist_ok=True)

print(f"CSV 경로: {CSV_PATH}")
print(f"PDF 출력 경로: {OUTPUT_DIR}")
print(f"체크포인트 경로: {CHECKPOINT_PATH}")
print(f"상세 페이지 동시 처리 개수: {DETAIL_CONCURRENCY}")
print(f"PDF 동시 처리 개수: {PDF_CONCURRENCY}")
print("차단 감지 시: 체크포인트 저장 후 수동 VPN 변경 -> 셀 재실행")


In [ ]:
def load_initial_targets() -> tuple[list[dict[str, str]], list[dict[str, object]], list[dict[str, object]], bool]:
    if CHECKPOINT_PATH.exists():
        payload = json.loads(CHECKPOINT_PATH.read_text(encoding="utf-8"))
        pending_rows = payload.get("pending_rows", [])
        for fallback_index, row in enumerate(pending_rows, start=1):
            row.setdefault("source_index", fallback_index)
        results = payload.get("results", [])
        failed_rows = payload.get("failed_rows", [])
        return pending_rows, results, failed_rows, True

    with CSV_PATH.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        pending_rows = [
            {**row, "source_index": index}
            for index, row in enumerate(reader, start=1)
            if row.get("detail_url")
        ][:MAX_ITEMS]
    return pending_rows, [], [], False


def save_checkpoint(
    pending_rows: list[dict[str, str]],
    results: list[dict[str, object]],
    failed_rows: list[dict[str, object]],
) -> None:
    payload = {
        "pending_rows": pending_rows,
        "results": results,
        "failed_rows": failed_rows,
    }
    CHECKPOINT_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


def clear_checkpoint() -> None:
    if CHECKPOINT_PATH.exists():
        CHECKPOINT_PATH.unlink()


def is_blocked_error(message: str | None) -> bool:
    if not message:
        return False
    markers = [
        "ERR_CONNECTION_CLOSED",
        "ERR_CONNECTION_RESET",
        "ERR_HTTP2_PROTOCOL_ERROR",
        "ERR_TIMED_OUT",
        "performance record timeout",
    ]
    return any(marker in message for marker in markers)


In [ ]:
pending_rows, results, failed_rows, resumed_from_checkpoint = load_initial_targets()
print(f"체크포인트 재개 여부: {resumed_from_checkpoint}")
print(f"남은 대상 개수: {len(pending_rows)}")
print(f"기존 성공 개수: {len(results)}")
print(f"기존 실패 개수: {len(failed_rows)}")
pending_rows[:2]


In [ ]:
detail_semaphore = asyncio.Semaphore(DETAIL_CONCURRENCY)
pdf_semaphore = asyncio.Semaphore(PDF_CONCURRENCY)

async def process_one(row: dict[str, str]) -> dict[str, object]:
    detail_url = row["detail_url"]
    item_started_at = time.perf_counter()
    try:
        async with detail_semaphore:
            detail_page = await crawl_detail_page(detail_url)

        async with pdf_semaphore:
            pdf_path = await download_performance_record_pdf(
                detail_url,
                OUTPUT_DIR,
                retry_count=PDF_RETRY_COUNT,
                retry_backoff_seconds=PDF_RETRY_BACKOFF_SECONDS,
                raise_on_blocked_error=True,
            )

        item_elapsed = time.perf_counter() - item_started_at
        return {
            "status": "ok",
            "order_index": row["source_index"],
            "detail_url": detail_url,
            "registration_number": detail_page.registration_number,
            "detail_page": detail_page.to_dict(),
            "performance_record_pdf": str(pdf_path) if pdf_path else None,
            "elapsed_seconds": round(item_elapsed, 2),
            "error": None,
            "blocked": False,
        }
    except Exception as exc:  # noqa: BLE001
        item_elapsed = time.perf_counter() - item_started_at
        error_message = str(exc)
        return {
            "status": "error",
            "order_index": row["source_index"],
            "detail_url": detail_url,
            "registration_number": None,
            "detail_page": None,
            "performance_record_pdf": None,
            "elapsed_seconds": round(item_elapsed, 2),
            "error": error_message,
            "blocked": is_blocked_error(error_message),
        }

completed_total = len(results) + len(failed_rows)
total_started_at = time.perf_counter()
chunk_started_at = total_started_at

while pending_rows:
    current_batch = pending_rows[:BATCH_SIZE]
    remaining_rows = pending_rows[BATCH_SIZE:]
    tasks = [asyncio.create_task(process_one(row)) for row in current_batch]

    batch_results = []
    blocked_rows = []

    for task in asyncio.as_completed(tasks):
        result = await task
        batch_results.append(result)
        completed_total += 1
        print(
            f"[{completed_total}] 완료 - 차량번호: {result.get('registration_number')} / PDF: {result.get('performance_record_pdf')} / 소요: {result.get('elapsed_seconds')}초 / 오류: {result.get('error')}"
        )

        if completed_total % LOG_EVERY == 0:
            chunk_elapsed = time.perf_counter() - chunk_started_at
            print(f"[로그] 최근 {LOG_EVERY}개 처리 시간: {chunk_elapsed:.2f}초")
            chunk_started_at = time.perf_counter()

    for row, result in zip(current_batch, sorted(batch_results, key=lambda item: item['order_index'])):
        if result['status'] == 'ok':
            results.append(result)
        elif result['blocked']:
            blocked_rows.append(row)
        else:
            failed_rows.append(result)

    pending_rows = blocked_rows + remaining_rows
    save_checkpoint(pending_rows, results, failed_rows)

    if len(blocked_rows) >= BLOCKED_ERROR_THRESHOLD:
        print("차단 의심 오류 감지. 여기서 중단합니다. VPN을 수동으로 바꾼 뒤 이 셀을 다시 실행하세요.")
        print("체크포인트는 이미 저장되었습니다.")
        break

if not pending_rows:
    clear_checkpoint()

if completed_total % LOG_EVERY != 0:
    chunk_size = completed_total % LOG_EVERY
    chunk_elapsed = time.perf_counter() - chunk_started_at
    print(f"[로그] 최근 {chunk_size}개 처리 시간: {chunk_elapsed:.2f}초")

total_elapsed = time.perf_counter() - total_started_at
print(f"전체 소요 시간: {total_elapsed:.2f}초")
print(f"남은 대상 개수: {len(pending_rows)}")
print(f"성공 개수: {len(results)}")
print(f"실패 개수: {len(failed_rows)}")


In [ ]:
print(json.dumps(results, ensure_ascii=False, indent=2))
print(json.dumps(failed_rows, ensure_ascii=False, indent=2))
print(pending_rows)
